TabPFN (Tabular Prior-Data Fitted Network) is a foundation model for tabular classification.

**It’s especially impressive for:**
- Small datasets (like < 10,000 samples)

**With TabPFN:**
- You don’t “train from scratch.”
- The model was pretrained offline
- On millions of synthetic tabular tasks
- Using Bayesian principles

So when you give it your dataset:
- It behaves like it has already “seen” similar problems
It’s like a GPT for tabular classification.

###### When is it NOT ideal?
❌ Very large datasets
❌ Huge number of samples (100k+)
❌ Regression (main version is classification)
❌ Heavy feature engineering pipelines

In [19]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

import os


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import os
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src.config import Config as Config  
from src.data_loader import load_data, prepare_data

cfg = Config

KAGGLE_EVAL = cfg.KAGGLE_EVAL
RANDOM_STATE = cfg.RANDOM_STATE
TASK = cfg.TASK
USE_POSTPROCESSING = cfg.USE_POSTPROCESSING
TARGET = cfg.TARGET
ID = cfg.ID
SUB_PATH = cfg.SUB_PATH
SUBMIT_PROBABILITIES = cfg.SUBMIT_PROBABILITIES


from src.data_loader import load_data, prepare_data
# from src.optuna_utils import run_optuna
from src.evaluation_utils import evaluate_model, evaluate_metric
# from src.visualization_utils import plot_feature_importance, plot_learning_curve, shap_summary
from src.postprocessing_utils import optimize_postprocessing, apply_postprocessing
from src.data_splitter import DataSplitter
from src.experiment_tracker import ExperimentTracker
from sklearn.utils import compute_class_weight

# -------------------------------------------------------

# -------------------------------
# Load & prepare data
# -------------------------------
X_train, X_test, y_train, y_test = load_data("encoded")
X_train, X_test, y_train_numeric, y_test_numeric, test_ids, num_classes, int_to_label = prepare_data(
    X_train, X_test, y_train, y_test, target=cfg.TARGET, drop_id=True
)

y_train_numeric = np.array(y_train_numeric)
y_test_numeric = np.array(y_test_numeric) if y_test_numeric is not None else None

# from sklearn.model_selection import train_test_split



from sklearn.model_selection import train_test_split


X_train, _, y_train_numeric, _ = train_test_split(
    X_train,
    y_train_numeric,
    test_size=0.95,              # keep only 10%
    random_state=RANDOM_STATE,
    stratify=y_train_numeric    # IMPORTANT: preserves class balance
)

print("Original:", X_train.shape)
print("Sample:", X_train.shape)

X_test = X_test.sample(
    frac=0.07,
    random_state=RANDOM_STATE
)

# -------------------------------
# Data splitter
# -------------------------------
splitter = DataSplitter(
    method="stratified_kfold",
    n_splits=5,
    random_state=RANDOM_STATE,
    folds_path="data/folds.npy"
)
folds = list(splitter.split(X_train, y_train_numeric, reuse_folds=False, verbose=True))


sys.path contains: /home/ismail/x42
Number of classes: 2
X_train shape: (300000, 116)
X_test shape: (200000, 116)
y_train shape: (300000,)
y_test labels are not available
Original: (15000, 116)
Sample: (15000, 116)
✅ Generated 5 folds for current dataset
--- Splitting data ---
Method: stratified_kfold
Number of splits: 5
Random seeds: [42]
Dataset size: 15000
Total folds: 5

Fold 0: Train size=12000, Val size=3000
Fold 1: Train size=12000, Val size=3000
Fold 2: Train size=12000, Val size=3000
Fold 3: Train size=12000, Val size=3000
Fold 4: Train size=12000, Val size=3000


In [21]:
import torch
import gc

def clean_vram():
    # Delete model and predictions if they exist in local scope
    global classifier, oof_preds, test_preds
    
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
    # Optional: If you want to be extra sure, check usage
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

clean_vram()

Allocated: 968.78 MB
Reserved: 1046.00 MB


In [ ]:
from tabpfn import TabPFNClassifier
import time


# Initialize TabPFN
# N_ensemble_configurations=32 is a good balance for speed/accuracy in a 12h window
# classifier = TabPFNClassifier(device='cuda', N_ensemble_configurations=32)
classifier = TabPFNClassifier(device='cuda')

# Dictionary to store out-of-fold predictions
oof_preds = np.zeros((len(X_train), num_classes))
test_preds = np.zeros((len(X_test), num_classes))

# -------------------------------
# Training Loop
# -------------------------------
# -------------------------------
# Training Loop with Batching
# -------------------------------
BATCH_SIZE = 1000 
start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\n--- Running TabPFN Fold {fold_idx + 1} ---")
    
    X_f_train, X_f_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_f_train, y_f_val = y_train_numeric[train_idx], y_train_numeric[val_idx]

    # Downsample training context to 10k
    if len(X_f_train) > 10000:
        idx = np.random.choice(len(X_f_train), 10000, replace=False)
        X_f_train_sub, y_f_train_sub = X_f_train.iloc[idx], y_f_train[idx]
    else:
        X_f_train_sub, y_f_train_sub = X_f_train, y_f_train

    classifier.fit(X_f_train_sub, y_f_train_sub)
    
    # --- BATCHED OOF PREDICTIONS ---
    print(f"Predicting OOF in batches...")
    fold_probs = []
    for i in range(0, len(X_f_val), BATCH_SIZE):
        batch = X_f_val.iloc[i : i + BATCH_SIZE]
        fold_probs.append(classifier.predict_proba(batch))
    oof_preds[val_idx] = np.vstack(fold_probs)
    
    # --- BATCHED TEST PREDICTIONS ---
    print(f"Predicting Test in batches...")
    test_probs_fold = []
    for i in range(0, len(X_test), BATCH_SIZE):
        batch = X_test.iloc[i : i + BATCH_SIZE]
        test_probs_fold.append(classifier.predict_proba(batch))
    test_preds += np.vstack(test_probs_fold) / len(folds)

train_time = time.time() - start


# -------------------------------
# Evaluation
# -------------------------------
overall_score = evaluate_metric(y_train_numeric, oof_preds, task='binary', kaggle_eval=KAGGLE_EVAL)
print(f"\nTabPFN Overall OOF Score: {overall_score}")


--- Running TabPFN Fold 1 ---
Predicting OOF in batches...
Predicting Test in batches...

--- Running TabPFN Fold 2 ---
Predicting OOF in batches...
Predicting Test in batches...

--- Running TabPFN Fold 3 ---
Predicting OOF in batches...
Predicting Test in batches...

--- Running TabPFN Fold 4 ---
Predicting OOF in batches...
Predicting Test in batches...

--- Running TabPFN Fold 5 ---
Predicting OOF in batches...
Predicting Test in batches...


TypeError: evaluate_metric() missing 2 required positional arguments: 'task' and 'kaggle_eval'

In [29]:
import gc

# -------------------------------
# Execution Block (TabPFN Version)
# -------------------------------

# We already have oof_preds and test_preds from your previous loop.
# If you haven't run it yet, ensure the TabPFN loop code block is executed.

# Since we don't have an Optuna 'study', we'll simulate the requirements
best_params = {"device": "cuda", "batch_size": BATCH_SIZE}
cv_mean_score = overall_score # From your previous evaluation

print("\nRunning final inference on full training context...")
# TabPFN 'final_model' is just the classifier instance 
# because it uses the training data as context.
final_model = classifier 

# -------------------------------
# Evaluate on train set
# -------------------------------
print("---------------- Train Set --------------------")
# For TabPFN, we predict on X_train using X_train as context (Fit)
# To avoid over-fitting metrics, we use the OOF preds for 'train' reporting 
# to be consistent with the CatBoost CV approach.
train_preds_input = oof_preds

if cfg.TASK.lower() == "binary":
    train_preds_input = train_preds_input[:, 1]
else:
    train_preds_input = train_preds_input

metrics_df, y_pred_train_class = evaluate_model(y_train_numeric, train_preds_input, task=cfg.TASK)
print(metrics_df)
print("The mean CV score (OOF):", cv_mean_score)

if cfg.TASK.lower() == "binary":
    oof_pred_class = (oof_preds >= 0.5).astype(int)
else:  
    oof_pred_class = np.argmax(oof_preds, axis=1)

oof_score = evaluate_metric(
    y_true=y_train_numeric,
    y_input=oof_preds, 
    task=TASK,
    kaggle_eval=KAGGLE_EVAL
)
print(f"OOF {KAGGLE_EVAL}: {oof_score:.5f}")

# -------------------------------
# Evaluate on test set
# -------------------------------
# test_preds was already calculated in your loop via averaging folds.
if cfg.TASK.lower() == "binary":
    test_preds_input = test_preds[:, 1] if test_preds.ndim > 1 else test_preds
    y_pred_class = (test_preds_input >= 0.5).astype(int)
else:
    test_preds_input = test_preds
    y_pred_class = test_preds.argmax(axis=1)

# -------------------------------
# Optional post-processing
# -------------------------------
postprocessing_params = {}
if USE_POSTPROCESSING:
    pp_seeds = [RANDOM_STATE, 2026, 1234, 9999]
    pp_best_params, pp_best_seed, pp_best_score = optimize_postprocessing(
        oof_preds,
        y_train_numeric,
        task=TASK,
        kaggle_eval=KAGGLE_EVAL,
        n_trials=100,
        seeds=pp_seeds
    )

    oof_pred_opt = apply_postprocessing(oof_preds, pp_best_params, task=TASK, return_proba=True)
    oof_pred_class_opt = apply_postprocessing(oof_preds, pp_best_params, task=TASK)
    oof_acc_opt = evaluate_metric(
        y_true=y_train_numeric,
        y_input=oof_pred_opt,
        task=TASK,
        kaggle_eval=KAGGLE_EVAL
    )

    print(f"OOF {KAGGLE_EVAL} after post-processing: {oof_acc_opt:.5f}")
    postprocessing_params = {
        "best_params": pp_best_params,
        "best_seed": pp_best_seed,
        "best_score": pp_best_score,
        "oof_pred_class_opt": oof_pred_class_opt,
        "oof_pred_opt": oof_pred_opt,
        "oof_acc_opt": oof_acc_opt,
        "kaggle_eval": KAGGLE_EVAL
    }

# -------------------------------
# ExperimentTracker
# -------------------------------
tracker = ExperimentTracker()
exp_dir = tracker.run_experiment(
    model_name="tabpfn",
    final_model=final_model, # The TabPFN instance
    X_train=X_train,
    y_train=y_train_numeric,
    X_test=X_test,
    best_params=best_params,
    best_score=cv_mean_score,
    metrics_df=metrics_df,
    train_time=train_time,
    oof_preds=oof_preds,
    task=TASK,
    postprocessing_params=postprocessing_params,
    use_postprocessing=USE_POSTPROCESSING,
    test_ids=test_ids,
    id_col=ID,
    target_col=TARGET,
    int_to_label=int_to_label,
    sample_submission_path=SUB_PATH,
    submit_proba=SUBMIT_PROBABILITIES
)

gc.collect()


Running final inference on full training context...
---------------- Train Set --------------------
     Metric     Value
0   ROC AUC  0.793703
1  Log Loss  0.492138
2  Accuracy  0.757400
The mean CV score (OOF): 0.793703251244875
OOF auc: 0.79370
Saved model → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/model.pkl
Saved params → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/params.json
Saved metrics → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/metrics.json
Saved training time → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/training_time.txt
Saved metadata → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/metadata.json
Updated experiments summary → /home/ismail/x42/outputs/experiments/experiments_summary.csv


/home/ismail/x42/src/experiment_tracker.py:451: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


Saved train predictions to /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/train_preds.npy
Saved test predictions to /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/test_preds.npy
Saved oof predictions to /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/oof_preds.npy
Saved Kaggle submission → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_/submission_raw_20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_.csv

✅ Experiment completed → /home/ismail/x42/outputs/experiments/20260423-115345_tabpfn_CVScore0.7937_exp_20260423-1153_



863

In [ ]:
import numpy as np

import plotly.figure_factory as ff
import plotly.express as px

from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform


pearson_corr = X_train.corr(method="pearson")

dissimilarity = 1 - np.abs(pearson_corr)

FONT_COLOR = "black"
BACKGROUND_COLOR = "white"
fig = ff.create_dendrogram(
    dissimilarity,
    labels=pearson_corr.columns,
    orientation="left",
    colorscale=px.colors.sequential.YlGnBu_r,
    # squareform() returns lower triangular in compressed form - as 1D array.
    linkagefun=lambda x: linkage(squareform(dissimilarity), method="complete"),
)
fig.update_layout(
    font_color=FONT_COLOR,
    title="Hierarchical Clustering using Correlation Matrix (Pearson)",
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    height=1340,
    width=840,
    yaxis=dict(
        showline=False,
        title="Feature",
        ticks="",
    ),
    xaxis=dict(
        showline=False,
        title="Distance",
        ticks="",
        range=[-0.05, 1.05],
    ),
)
fig.update_traces(line_width=1.5)
fig.show()

/home/ismail/miniconda/envs/ml-dl/lib/python3.10/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




Okay, here we need to make something clear. Since we had the correlation matrix, we conducted hierarchical clustering. 
This process consists of an alternative to the K-Means algorithm. Hierarchical clustering allows us to visualize the effect of different clusters' number determining.
However, relying on a correlation matrix to perform hierarchical clustering requires additional steps. Primarily, clustering methods measure the dissimilarity of variables. Meanwhile, correlation measures similarity. We can treat dissimilarity as dissimilarity=1−abs(correlation)
. And basically, that's all. We passed dissimilarity to the linkage() function from the scipy module and got clustering results.
Moreover, we should remember that we rely on the Pearson correlation. It measures linear dependency, and it's computed on actual values. However, we could have used for example the Spearman correlation, which is based on ranks and measures monotonic relations.
Additionally, we chose the complete method in the linkage() function, and if you take a different method, you get different results.
As you can see, here we have minimal distances between BZ - BC, DV - CL, and EH - FD.

**For each cluster you can create aggregated features:**

- cluster_mean
- cluster_std
- cluster_max


In [31]:
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from lightgbm import LGBMClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

casual_preprocess = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


lda_pipeline = make_pipeline(
    casual_preprocess,
    LinearDiscriminantAnalysis(),
).fit(X_train, y_train)
lda_info = np.abs(lda_pipeline[-1].scalings_.ravel())
lda_info = lda_info / lda_info.sum()  # Normalise to 1 to compare with other methods.

lgbm_pipeline = make_pipeline(
    casual_preprocess,
    LGBMClassifier(random_state=42, is_unbalance=True),
).fit(X_train, y_train)
lgbm_info = lgbm_pipeline[-1].feature_importances_
lgbm_info = lgbm_info / lgbm_info.sum()

mutual_info = mutual_info_classif(
    X=casual_preprocess.fit_transform(X_train), y=y_train, random_state=42
)
mutual_info = mutual_info / np.sum(mutual_info)

importances = pd.DataFrame(
    [lda_info, lgbm_info, mutual_info],
    columns=lda_pipeline[0].get_feature_names_out(),
    index=["LDA", "LGBM", "MI"],
).T

importances[:10]

ValueError: SimpleImputer does not support data with dtype bool. Please provide either a numeric array (with a floating point or integer dtype) or categorical data represented either as an array with integer dtype or an array of string values with an object dtype.

In [7]:
import plotly.express as px

color_map = "Viridis"  # or any other Plotly continuous color scale
FONT_COLOR = "black"
BACKGROUND_COLOR = "white"

importances_melted_frame = (
    importances.melt(
        var_name="Method",
        value_name="Importance",
        ignore_index=False,
    )
    .reset_index()
    .rename(columns={"index": "Feature"})
    .round(4)
)

fig = px.bar(
    importances_melted_frame,
    x="Importance",
    y="Feature",
    color="Importance",
    facet_col="Method",
    facet_col_spacing=0.07,
    height=940,
    width=840,
    color_continuous_scale=color_map,
    title="Normalised Feature Importances (Three Different Default Methods)",
)
fig.update_annotations(font_size=14)
fig.update_yaxes(
    matches=None,
    showticklabels=True,
    categoryorder="total ascending",
    tickfont_size=8,
)
fig.update_xaxes(matches=None)
fig.update_traces(width=0.7)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    coloraxis_colorbar=dict(
        orientation="h",
        title_side="bottom",
        yanchor="bottom",
        xanchor="center",
        title=None,
        y=-0.2,
        x=0.5,
    ),
)
fig.show()

##### permutation-based feature importance analysis

This code performs a permutation-based feature importance analysis for classification models using a custom balanced log loss metric.

Purpose / Benefits:

- Helps identify which features are truly important for the model.

- Provides interpretability for your ML models.

- Works well with imbalanced datasets due to the balanced log loss.

- Can guide feature selection and data collection priorities.


We will perform the so-called permutation test to see when the balanced log loss metric is mostly sensitive while permuting samples in a certain feature.

In [63]:
# ----------------------------
# Imports
# ----------------------------
import numpy as np
import pandas as pd
from functools import partial
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import plotly.express as px

# ----------------------------
# Placeholder for preprocessing
# ----------------------------
# Replace this with your actual preprocessing pipeline
casual_preprocess = make_pipeline(
    SimpleImputer(strategy="median"),  # Fill NaNs with median
    StandardScaler()                   # Scale features
)

# ----------------------------
# Colors and theme
# ----------------------------
color_map = "viridis"
FONT_COLOR = "black"
BACKGROUND_COLOR = "white"

# ----------------------------
# Balanced log loss function
# ----------------------------
def balanced_log_loss(y_true, y_pred, eps=1e-15):
    """Balanced log loss for binary classification."""
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = np.asarray(y_pred).ravel()

    N0, N1 = np.bincount(y_true)
    y0 = (y_true == 0).astype(int)
    y1 = (y_true == 1).astype(int)

    y_pred = np.clip(y_pred, eps, 1 - eps)
    p0 = np.log(1 - y_pred)
    p1 = np.log(y_pred)

    return -0.5 * (np.sum(y0 * p0) / N0 + np.sum(y1 * p1) / N1)

# ----------------------------
# Parameters
# ----------------------------
n_bags = 10
n_folds = 5
np.random.seed(42)
seeds = np.random.randint(0, 19937, size=n_bags)

# ----------------------------
# Classifiers
# ----------------------------
forest = RandomForestClassifier(class_weight="balanced", criterion="log_loss", random_state=42)
svc = SVC(class_weight="balanced", probability=True, random_state=42)
lgbm = LGBMClassifier(is_unbalance=True, random_state=42)

# ----------------------------
# Storage
# ----------------------------
original_loglosses = []
permutation_loglosses = pd.DataFrame()

# ----------------------------
# Loop over classifiers
# ----------------------------
for classifier in (forest, svc, lgbm):
    y_proba_original = np.zeros(len(y_train), dtype=np.float64)
    y_proba_shuffled = defaultdict(lambda: np.zeros(len(y_train), dtype=np.float64))

    for seed in seeds:
        skfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        classifier.set_params(random_state=seed)

        for train_idx, valid_idx in skfold.split(X_train, y_train):
            X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
            X_val, y_val = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

            # Preprocess
            X_tr_proc = casual_preprocess.fit_transform(X_tr)
            X_val_proc = casual_preprocess.transform(X_val)

            # Fit classifier
            classifier.fit(X_tr_proc, y_tr)
            y_proba_original[valid_idx] += classifier.predict_proba(X_val_proc)[:, 1]

            # Permutation
            for i, feature in enumerate(casual_preprocess.get_feature_names_out() if hasattr(casual_preprocess, 'get_feature_names_out') else X_train.columns):
                X_val_shuffled = X_val_proc.copy()
                X_val_shuffled[:, i] = np.random.permutation(X_val_shuffled[:, i])
                y_proba_shuffled[feature][valid_idx] += classifier.predict_proba(X_val_shuffled)[:, 1]

    # ----------------------------
    # Compute balanced log loss
    # ----------------------------
    classifier_name = classifier.__class__.__name__
    feature_names = list(y_proba_shuffled.keys())

    original_loglosses.append(balanced_log_loss(y_train, y_proba_original / n_bags))
    permutation_loglosses[classifier_name] = pd.Series(
        [balanced_log_loss(y_train, y_proba_shuffled[feat] / n_bags) for feat in feature_names],
        index=feature_names
    )

# ----------------------------
# Melt dataframe for Plotly
# ----------------------------
permutation_results_melted = (
    permutation_loglosses.melt(
        var_name="Method",
        value_name="Balanced Log Loss",
        ignore_index=False,
    )
    .reset_index()
    .rename(columns={"index": "Feature"})
    .round(4)
)

# ----------------------------
# Plot
# ----------------------------
fig = px.bar(
    permutation_results_melted,
    x="Balanced Log Loss",
    y="Feature",
    color="Balanced Log Loss",
    facet_col="Method",
    facet_col_spacing=0.07,
    height=940,
    width=840,
    color_continuous_scale=color_map,
    title="Permutation Test Results - Balanced Log Loss",
)
fig.update_annotations(font_size=14)
fig.update_traces(width=0.7)
fig.update_xaxes(matches=None)
fig.update_yaxes(
    matches=None,
    showticklabels=True,
    categoryorder="total ascending",
    tickfont_size=8,
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    coloraxis_colorbar=dict(
        orientation="h",
        title_side="bottom",
        yanchor="bottom",
        xanchor="center",
        title=None,
        y=-0.2,
        x=0.5,
    ),
    margin_t=120,
)

# Add vertical lines for original logloss
for original_logloss, max_logloss, col in zip(original_loglosses, permutation_loglosses.max().tolist(), (1, 2, 3)):
    fig.add_vline(
        x=original_logloss,
        line_width=2,
        line_dash="dash",
        line_color="#FF2079",
        col=col,
    )
    fig.add_vrect(
        x0=original_logloss,
        x1=max_logloss,
        line_width=0,
        fillcolor="#FF2079",
        opacity=0.2,
        col=col,
    )

fig.show()

In [64]:


greeks = greeks.reset_index(drop=True).join(y_train.reset_index(drop=True))
greeks_cats = greeks[["Alpha", "Beta", "Gamma", "Delta"]].astype("category")
greeks_codes = greeks_cats.apply(lambda x: x.cat.codes)



In [65]:
y_train

,Class
0,1
1,0
2,0
3,0
4,1
...,...
612,0
613,0
614,0
615,0


**parallel coordinates plot**

In [66]:
import plotly.graph_objects as go

fig = go.Figure(
    go.Parcoords(
        dimensions=[
            dict(
                label="Beta",
                values=greeks_codes.Beta,
                tickvals=np.unique(greeks_codes.Beta),
                ticktext=greeks_cats.Beta.cat.categories,
            ),
            dict(
                label="Gamma",
                values=greeks_codes.Gamma,
                tickvals=np.unique(greeks_codes.Gamma),
                ticktext=greeks_cats.Gamma.cat.categories,
            ),
            dict(
                label="Delta",
                values=greeks_codes.Delta,
                tickvals=np.unique(greeks_codes.Delta),
                ticktext=greeks_cats.Delta.cat.categories,
            ),
            dict(
                label="Alpha",
                values=greeks_codes.Alpha,
                tickvals=np.unique(greeks_codes.Alpha),
                ticktext=greeks_cats.Alpha.cat.categories,
            ),
            dict(
                label="Class",
                values=greeks.Class,
                tickvals=np.unique(greeks.Class),
            ),
        ],
        line=dict(
            color=greeks.Class,
            colorscale=color_map,
            showscale=True,
            colorbar=dict(
                title="Class",
                orientation="h",
                title_side="bottom",
                yanchor="bottom",
                xanchor="center",
                y=-0.35,
                x=0.5,
                nticks=2,
            ),
        ),
    )
)

fig.update_layout(
    font_color=FONT_COLOR,
    title="Greeks - Parallel Coordinates",
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    height=540,
    width=840,
)
fig.update_traces(
    labelfont=dict(family="Arial Black", size=10),
    tickfont=dict(family="Arial Black", size=10),
    selector=dict(type="parcoords"),
)
fig.show()



In [9]:
num_cols = X_train.select_dtypes(include=['float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

In [12]:
import lightgbm as lgb
import xgboost as xgb
from catboost import Pool, CatBoostRegressor, CatBoostClassifier



In [21]:
from pathlib import Path

class CFG:
    VER = 1
    AUTHOR = 'maverick'
    COMPETITION = 'icr-identify-age-related-conditions'
    # DATA_PATH = path('/kaggle/input/icr-identify-age-related-conditions')
    OOF_DATA_PATH = Path('../outputs/oof_preds')
    MODEL_DATA_PATH = Path('../outputs/models')
    METHOD_LIST = ['lightgbm', 'xgboost', 'catboost']
    seed = 3407 #52
    n_folds = 10 #replaced 20
    target_col = 'Class'
    metric = 'balanced_log_loss'
    metric_maximize_flag = False
    num_boost_round = 50500
    early_stopping_round = 500
    verbose = 2000
    boosting_type = 'dart'
    lgb_params = {
        'objective': 'binary', # 'binary', 'multiclass'
        'metric': None, # 'auc', 'multi_logloss'
        # 'num_class': None,
        'boosting': boosting_type,
        'device_type':'gpu',
        'learning_rate': 0.005,
        'num_leaves': 5,
        'feature_fraction': 0.50,
        'bagging_fraction': 0.80,
        'lambda_l1': 2, 
        'lambda_l2': 4,
        'n_jobs': -1,
        'is_unbalance':True, #added balancing
        'verbose': -1, #added silence
        # 'min_data_in_leaf': 40,
        # 'bagging_freq': 10,
        'seed': seed,
    }
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'learning_rate': 0.005, 
        'max_depth': 4,
        'colsample_bytree': 0.50,
        'subsample': 0.80,
        'eta': 0.03,
        'gamma': 1.5,
        # 'lambda': 70,
        # 'min_child_weight': 8,
        # 'eval_metric':'logloss',
        # 'tree_method': 'gpu_hist',
        # 'predictor':'gpu_predictor',
        'random_state': seed,
    }
    
    cat_params = {
        'learning_rate': 0.005, 
        'iterations': num_boost_round, 
        'depth': 4, # 
        'colsample_bylevel': 0.50,
        'subsample': 0.80,
        'l2_leaf_reg': 3, # 3, 30
        'random_seed': seed,
        'auto_class_weights': 'Balanced'
        # 'task_type':'GPU', 
    }

In [16]:


X_test[CFG.target_col] = -1
all_df = pd.concat([X_train, X_test])



In [18]:
import random


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['ml-dl'] = str(seed)

In [19]:
def balanced_log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    nc = np.bincount(y_true)
    w0, w1 = 1/(nc[0]/y_true.shape[0]), 1/(nc[1]/y_true.shape[0])
    balanced_log_loss_score = (-w0/nc[0]*(np.sum(np.where(y_true==0,1,0) * np.log(1-y_pred))) - w1/nc[1]*(np.sum(np.where(y_true!=0,1,0) * np.log(y_pred)))) / (w0+w1)
    return balanced_log_loss_score

# ====================================================
# LightGBM Metric
# ====================================================
def lgb_metric(y_pred, y_true):
    y_true = y_true.get_label()
    return 'balanced_log_loss', balanced_log_loss(y_true, y_pred), CFG.metric_maximize_flag

def xgb_metric(y_pred, y_true):
    y_true = y_true.get_label()
    return 'balanced_log_loss', balanced_log_loss(y_true, y_pred)


# ====================================================
# Catboost Metric
# ====================================================
class CatboostMetric(object):
    def get_final_error(self, error, weight): return error
    def is_max_optimal(self): return CFG.metric_maximize_flag
    def evaluate(self, approxes, target, weight):
        error = balanced_log_loss(np.array(target), approxes)
        return error, 0

In [37]:
import gc
import pickle
from sklearn.model_selection import StratifiedKFold

def calc_log_loss_weight(y_true):
    nc = np.bincount(y_true)
    w0, w1 = 1/(nc[0]/y_true.shape[0]), 1/(nc[1]/y_true.shape[0])
    return w0, w1
def lightgbm_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
    train_w0, train_w1 = calc_log_loss_weight(y_train)
    valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
    lgb_train = lgb.Dataset(x_train, y_train, weight=y_train.map({0: train_w0, 1: train_w1}), categorical_feature=categorical_features)
    lgb_valid = lgb.Dataset(x_valid, y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1}), categorical_feature=categorical_features)
    model = lgb.train(
                params = CFG.lgb_params,
                train_set = lgb_train,
                num_boost_round = CFG.num_boost_round,
                valid_sets = [lgb_train, lgb_valid],
                callbacks=[
                    lgb.early_stopping(CFG.early_stopping_round),
                    lgb.log_evaluation(CFG.verbose)
                ]
                # feval = lgb_metric,
            )
    # Predict validation
    valid_pred = model.predict(x_valid)
    return model, valid_pred


def xgboost_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
    train_w0, train_w1 = calc_log_loss_weight(y_train)
    valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
    xgb_train = xgb.DMatrix(data=x_train, label=y_train, weight=y_train.map({0: train_w0, 1: train_w1}))
    xgb_valid = xgb.DMatrix(data=x_valid, label=y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1}))
    model = xgb.train(
                CFG.xgb_params, 
                dtrain = xgb_train, 
                num_boost_round = CFG.num_boost_round, 
                evals = [(xgb_train, 'train'), (xgb_valid, 'eval')], 
                early_stopping_rounds = CFG.early_stopping_round, 
                verbose_eval = CFG.verbose,
                # feval = xgb_metric, 
                # maximize = CFG.metric_maximize_flag, 
            )
    # Predict validation
    valid_pred = model.predict(xgb.DMatrix(x_valid), iteration_range=(0, model.best_ntree_limit))
    return model, valid_pred


def catboost_training(x_train: pd.DataFrame, y_train: pd.DataFrame, x_valid: pd.DataFrame, y_valid: pd.DataFrame, features: list, categorical_features: list):
    train_w0, train_w1 = calc_log_loss_weight(y_train)
    valid_w0, valid_w1 = calc_log_loss_weight(y_valid)
    cat_train = Pool(data=x_train, label=y_train, weight=y_train.map({0: train_w0, 1: train_w1}), cat_features=categorical_features)
    cat_valid = Pool(data=x_valid, label=y_valid, weight=y_valid.map({0: valid_w0, 1: valid_w1}), cat_features=categorical_features)
    model = CatBoostClassifier(**CFG.cat_params) # , eval_metric = CatboostMetric
    model.fit(cat_train, 
              eval_set=[cat_valid],
              early_stopping_rounds=CFG.early_stopping_round, 
              verbose=CFG.verbose, 
              use_best_model=True)
    # Predict validation
    valid_pred = model.predict_proba(x_valid)[:, 1]
    return model, valid_pred


def gradient_boosting_model_cv_training(method: str, train_df: pd.DataFrame, features: list, categorical_features: list):
    # Create a numpy array to store out of folds predictions
    oof_predictions = np.zeros(len(train_df))
    oof_fold = np.zeros(len(train_df))
    kfold = StratifiedKFold(n_splits = CFG.n_folds, shuffle = True, random_state = CFG.seed)
    for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.target_col])):
        print('-'*50)
        print(f'{method} training fold {fold + 1}')
        
        x_train = train_df[features].iloc[train_index]
        y_train = train_df[CFG.target_col].iloc[train_index]
        x_valid = train_df[features].iloc[valid_index]
        y_valid = train_df[CFG.target_col].iloc[valid_index]
        if method == 'lightgbm':
            model, valid_pred = lightgbm_training(x_train, y_train, x_valid, y_valid, features, categorical_features)
        if method == 'xgboost':
            model, valid_pred = xgboost_training(x_train, y_train, x_valid, y_valid, features, categorical_features)
        if method == 'catboost':
            model, valid_pred = catboost_training(x_train, y_train, x_valid, y_valid, features, categorical_features)
        
        # Save best model
        pickle.dump(model, open(CFG.MODEL_DATA_PATH / f'{method}_fold{fold + 1}_seed{CFG.seed}_ver{CFG.VER}.pkl', 'wb'))
        # Add to out of folds array
        oof_predictions[valid_index] = valid_pred
        oof_fold[valid_index] = fold + 1
        del x_train, x_valid, y_train, y_valid, model, valid_pred
        gc.collect()

    # Compute out of folds metric
    score = balanced_log_loss(train_df[CFG.target_col], oof_predictions)
    print(f'{method} our out of folds CV score is {score}')
    # Create a dataframe to store out of folds predictions
    oof_df = pd.DataFrame({'Id': train_df['Id'], CFG.target_col: train_df[CFG.target_col], f'{method}_prediction': oof_predictions, 'fold': oof_fold})
    oof_df.to_csv(CFG.MODEL_DATA_PATH / f'oof_{method}_seed{CFG.seed}_ver{CFG.VER}.csv', index = False)


In [38]:
numerical_features = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', #'BC', 
                      'BD', 'BN', 'BP', 'BQ', 'BR', 'BZ',
                      'CB', 'CC', 'CD', 'CF', 'CH', #'CL', 
                      'CR', 'CS', 'CU', 'CW',
                      'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
                      'EB', 'EE', 'EG', 'EH', 'EL', 'EP', 'EU',
                      'FC', 'FD', 'FE', 'FI', 'FL', 'FR', 'FS',
                      'GB', 'GE', 'GF', 'GH', 'GI', 'GL']

categorical_features = []
features = numerical_features + categorical_features
seed_everything(CFG.seed)

In [39]:
def Preprocessing(input_df: pd.DataFrame)->pd.DataFrame:
    input_df = input_df.rename(columns={'BD_': 'BD', 'CD_': 'CD', 'CW_': 'CW', 'FD_': 'FD'}) #?
    output_df = input_df.copy()
    sc = StandardScaler()
    output_df[numerical_features] = sc.fit_transform(output_df[numerical_features]) #added scaling
    return output_df
all_df = Preprocessing(all_df)

In [40]:


train_df = all_df[all_df[CFG.target_col] != -1].copy()
test_df = all_df[all_df[CFG.target_col] == -1].copy()



In [41]:
train_df[CFG.target_col] = y_train
train_df

,AB,AF,AH,AM,AR,AX,AY,AZ,BC,BD,...,GB,GE,GF,GH,GI,GL,EJ_A,EJ_B,Id,Class
0,-0.563887,-0.157903,-0.254351,-0.234043,-0.181606,-1.855732,-0.082579,-0.151040,5.555634,-0.387852,...,-0.911489,-0.403187,-0.650529,-0.891200,0.540440,-0.809484,0.0,1.0,NaN,1
1,-0.700822,-1.079913,-0.254351,-0.024272,-0.181606,-0.722330,-0.082579,0.685200,1.229900,0.062261,...,-1.113819,-0.403187,0.695193,-0.205767,-0.496139,1.311539,1.0,0.0,NaN,0
2,-0.007016,-0.363025,-0.254351,-0.090600,-0.181606,0.476131,-0.082579,0.528761,1.229900,-0.056365,...,1.633275,-0.292061,-0.045827,-0.314722,-0.411991,-0.802051,0.0,1.0,NaN,0
3,-0.472596,0.149663,0.019823,0.553510,-0.181606,-0.701785,-0.082579,0.129129,1.229900,-0.373698,...,-0.200574,-0.335076,-0.645843,0.853176,1.108405,-0.806041,0.0,1.0,NaN,0
4,-0.198725,0.112179,-0.254351,-0.353370,-0.181606,-0.602484,-0.012113,-1.598817,102.151980,0.138401,...,-0.410262,0.107343,-0.312745,1.385198,-0.382574,-0.811787,0.0,1.0,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
612,-0.691693,-0.148803,0.047725,-0.419430,0.283051,-0.773693,0.042205,-0.436898,2.804172,-0.377637,...,-1.146008,0.600785,-0.334940,-0.645960,0.522762,1.311539,1.0,0.0,NaN,0
613,-0.080048,0.860509,-0.254351,0.113643,0.564042,0.185076,-0.081845,0.483251,3.777550,0.113878,...,1.516475,2.544620,-0.594504,-0.155480,2.051855,-0.807059,0.0,1.0,NaN,0
614,-0.098306,-0.439200,0.097662,0.240370,-0.003928,0.993181,-0.082579,1.111854,1.229900,0.191003,...,-0.058943,-0.012220,-0.418927,-0.496557,1.907530,1.311539,1.0,0.0,NaN,0
615,-0.235241,-0.956660,-0.254351,-0.215455,-0.181606,0.958939,-0.082579,-0.667291,1.229900,-0.259329,...,0.399060,-0.403187,-0.652522,-0.599317,-0.358037,-0.803247,0.0,1.0,NaN,0


In [42]:

for method in CFG.METHOD_LIST:
    gradient_boosting_model_cv_training(method, train_df, features, categorical_features)



--------------------------------------------------
lightgbm training fold 1
[2000]	training's binary_logloss: 0.239171	valid_1's binary_logloss: 0.383965
[4000]	training's binary_logloss: 0.133602	valid_1's binary_logloss: 0.333064
[6000]	training's binary_logloss: 0.0832013	valid_1's binary_logloss: 0.313513
[8000]	training's binary_logloss: 0.0550198	valid_1's binary_logloss: 0.315459
[10000]	training's binary_logloss: 0.0412783	valid_1's binary_logloss: 0.326878
[12000]	training's binary_logloss: 0.0318485	valid_1's binary_logloss: 0.335052
[14000]	training's binary_logloss: 0.0268753	valid_1's binary_logloss: 0.344306
[16000]	training's binary_logloss: 0.0230512	valid_1's binary_logloss: 0.355845
[18000]	training's binary_logloss: 0.0209145	valid_1's binary_logloss: 0.361629
[20000]	training's binary_logloss: 0.0193517	valid_1's binary_logloss: 0.366238
[22000]	training's binary_logloss: 0.0181072	valid_1's binary_logloss: 0.368689
[24000]	training's binary_logloss: 0.0172699	valid

AttributeError: 'Booster' object has no attribute 'best_ntree_limit'

In [67]:
def get_undersampling_fraction(y_true):
    N0, N1 = np.bincount(y_true)
    return 1 - N1 / N0


def assert_balanced_learning(y_train, n_samples_tol=1):
    N0, N1 = np.bincount(y_train)
    assert np.isclose(N0, N1, atol=n_samples_tol)


def get_sample_weights(y_true, weights=None):
    """Pass `weights` tuple as `(weight_class_0, weight_class_1)`
    if you want to use custom weights."""
    N0, N1 = np.bincount(y_true)
    y0, y1 = np.unique(y_true)

    if weights:
        w0, w1 = weights
        return np.where(y_true == y1, w1, w0)

    w0 = (N0 + N1) / N0
    w1 = (N0 + N1) / N1

    return np.where(y_true == y1, w1, w0)


def perform_proba_postprocessing(
    y_proba,
    rounding=True,
    rounding_prec=4,
    boosting=True,
    boosting_coef=0.8,
    shifting=True,
    shifting_map=None,
):
    """Fancy postprocessing. Highly probable that do nothing or deteriorates."""

    def my_ceil(x, prec=rounding_prec):
        return np.true_divide(np.ceil(x * 10**prec), 10**prec)

    def my_floor(x, prec=rounding_prec):
        return np.true_divide(np.floor(x * 10**prec), 10**prec)

    proba = y_proba.copy()

    if rounding:
        proba = np.where(proba > 0.5, my_floor(proba), my_ceil(proba))

    if boosting:
        odds = boosting_coef * proba / (1 - proba)
        proba = odds / (1 + odds)

    if shifting:
        if not shifting_map:
            shifting_map = {"low": (0.01, 0.02), "high": (0.99, 0.98)}
        low_shift_from, low_shift_to = shifting_map.get("low", (0.01, 0.02))
        high_shift_from, high_shift_to = shifting_map.get("high", (0.99, 0.98))
        proba[proba < low_shift_from] = low_shift_to
        proba[proba > high_shift_from] = high_shift_to

    return proba

In [68]:
n_bags = 20
n_folds = 10

np.random.seed(42)
seeds = np.random.randint(0, 19937, size=n_bags)

X = X_train
y = y_train.Class

lgbm_params = {
    "max_depth": 4,
    "num_leaves": 9,
    "min_child_samples": 17,
    "n_estimators": 200,
    "learning_rate": 0.15,
    "colsample_bytree": 0.4,
    "min_split_gain": 1e-4,
    "reg_alpha": 1e-2,
    "reg_lambda": 5e-3,
}

xgb_params = {
    "max_depth": 2,
    "n_estimators": 200,
    "learning_rate": 0.4,
    "subsample": 0.6,
    "min_child_weight": 0.1,
    "max_delta_step": 0.35,
    "colsample_bytree": 0.3,
    "colsample_bylevel": 0.7,
    "min_split_loss": 1e-4,
    "reg_alpha": 2e-3,
    "reg_lambda": 6e-2,
}

svc_params = {
    "probability": True,
    "C": 3,
}

In [69]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier


undersampling_frac = get_undersampling_fraction(y)
y_proba = np.zeros_like(y, dtype=np.float64)
classifiers = defaultdict(object)

for bag, seed in enumerate(seeds):
    skfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for fold, (train_ids, valid_ids) in enumerate(skfold.split(X, y)):
        y_train_full = y.iloc[train_ids]
        to_undersample_ids = (
            y_train_full[y_train_full == 0]
            .sample(frac=undersampling_frac, random_state=seed)
            .index.to_numpy()
        )
        # Skfold returns numbers, but `y` is a series with IDs, so we map them.
        to_undersample_ids = [y.index.get_loc(idx) for idx in to_undersample_ids]
        train_ids = np.setdiff1d(train_ids, to_undersample_ids)

        X_train, y_train = X.iloc[train_ids], y.iloc[train_ids]
        X_valid, y_valid = X.iloc[valid_ids], y.iloc[valid_ids]

        assert_balanced_learning(y_train)

        current_ensemble = make_pipeline(
            casual_preprocess,
            VotingClassifier(
                [
                    ("lgbm", LGBMClassifier(random_state=seed, **lgbm_params)),
                    ("xgb", XGBClassifier(random_state=seed, **xgb_params)),
                    ("svc", SVC(random_state=seed, **svc_params)),
                ],
                voting="soft",
                weights=(0.45, 0.45, 0.10),
            ),
        ).fit(X_train, y_train)

        y_proba[valid_ids] += current_ensemble.predict_proba(X_valid)[:, 1]
        classifiers[f"Voting Bag: {bag} Fold: {fold}"] = current_ensemble

y_proba_averaged = y_proba / n_bags

In [70]:
from sklearn.metrics import brier_score_loss


print("Balanced Log Loss:", f"{balanced_log_loss(y, y_proba_averaged):.5f}")
print("Brier Score Loss: ", f"{brier_score_loss(y, y_proba_averaged):.5f}")


Balanced Log Loss: 0.22201
Brier Score Loss:  0.06704


In [71]:
y_proba_postprocessed = perform_proba_postprocessing(y_proba_averaged)
print(
    "Postprocessed Balanced Log Loss:",
    f"{balanced_log_loss(y, y_proba_postprocessed):.5f}",
)
print(
    "Postprocessed Brier Score Loss: ",
    f"{brier_score_loss(y, y_proba_postprocessed):.5f}",
)


Postprocessed Balanced Log Loss: 0.21539
Postprocessed Brier Score Loss:  0.06103


In [72]:
y_proba_frame = pd.DataFrame(
    {
        "Sample Integer Index": np.arange(0, len(y)),
        "Positive Class Probability": y_proba_averaged,
        "Class": y.values.astype(str),
        
    },
    index=y.index,
)

fig = px.scatter(
    y_proba_frame.reset_index(),
    x="Positive Class Probability",
    y="Sample Integer Index",
    symbol="Class",
    symbol_sequence=["diamond", "circle"],
    color="Class",
    color_discrete_sequence=["#010D36", "#FF2079"],
    category_orders={"Class": ("0", "1")},
    # hover_data="Id",
    opacity=0.6,
    height=540,
    width=840,
    title="Training Dataset - Out of Fold Predictions",
)
fig.update_layout(
    font_color=FONT_COLOR,
    title_font_size=18,
    plot_bgcolor=BACKGROUND_COLOR,
    paper_bgcolor=BACKGROUND_COLOR,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        xanchor="right",
        y=1.05,
        x=1,
        title="Class",
        itemsizing="constant",
    ),
    xaxis_range=[-0.02, 1.02],
)
fig.update_traces(marker_size=6)
fig.show()

In [74]:
# Dummy protection for an empty test dataset.
if np.all(np.isclose(X_test.select_dtypes("number").sum(), 0)):
    X_test_numeric_cols = X_test.select_dtypes("number").columns
    X_test[X_test_numeric_cols] += 1e-9

test_ids = X_test.index
# y_test = np.zeros_like(test_ids)

X_test_features = X_test.drop(columns=["Id"], errors="ignore")  # drop Id if exists
X_test_features = X_test_features[X_train.columns]
y_test = np.zeros(len(X_test_features))

for classifier in classifiers.values():
    # Each classifier contains preprocessing, so we pass raw test dataset.
    y_test += classifier.predict_proba(X_test_features)[:, 1]

y_test_averaged = y_test / len(classifiers)

submission = pd.DataFrame(
    {
        "Id": test_ids,
        "class_0": 1 - y_test_averaged,
        "class_1": y_test_averaged,
    }
).set_index("Id")

submission.to_csv("../outputs/submission_ensamble.csv")
submission.head()

,class_0,class_1
Id,,
0,0.613545,0.386455
1,0.613545,0.386455
2,0.613545,0.386455
3,0.613545,0.386455
4,0.613545,0.386455


In [55]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

# Make sure X_train and y_train_numeric have the same number of rows
X_train = pd.read_csv('../data/X_train_encoded.csv')
# Split into train/validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_numeric, test_size=0.2, stratify=y_train_numeric, random_state=42
)

# Create datasets
lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_valid = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Parameters
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',  # or 'auc'
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'is_unbalance': True,  # handles imbalanced data
    'verbosity': -1
}

# Train using callbacks for early stopping and logging
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# Predictions
y_pred_proba = gbm.predict(X_val, num_iteration=gbm.best_iteration)
y_pred_class = (y_pred_proba >= 0.5).astype(int)

# Evaluation
logloss = log_loss(y_val, y_pred_proba)
auc = roc_auc_score(y_val, y_pred_proba)
print(f"Validation Log Loss: {logloss:.4f}")
print(f"Validation AUC: {auc:.4f}")

Training until validation scores don't improve for 50 rounds
[50]	train's binary_logloss: 0.0894696	valid's binary_logloss: 0.252485
[100]	train's binary_logloss: 0.0225644	valid's binary_logloss: 0.222118
[150]	train's binary_logloss: 0.00799046	valid's binary_logloss: 0.205056
[200]	train's binary_logloss: 0.00374326	valid's binary_logloss: 0.206186
Early stopping, best iteration is:
[179]	train's binary_logloss: 0.00504732	valid's binary_logloss: 0.203424
Validation Log Loss: 0.2034
Validation AUC: 0.9652


In [8]:
def random_under_sampler(X, y):
    """
    Random undersampling for binary classification
    """
    import pandas as pd
    import numpy as np

    # Convert X to DataFrame if needed
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    
    y = np.array(y).flatten()  # ensure 1D array

    # Count classes
    neg, pos = np.bincount(y)

    # Separate classes
    one_idx = np.where(y == 1)[0]
    zero_idx = np.where(y == 0)[0]

    # Randomly sample majority class
    zero_idx_sampled = np.random.choice(zero_idx, size=pos, replace=False)

    # Combine indices
    selected_idx = np.concatenate([one_idx, zero_idx_sampled])

    # Shuffle
    np.random.shuffle(selected_idx)

    # Return undersampled X and y
    return X.iloc[selected_idx], y[selected_idx]

X_train_bal, y_train_bal = random_under_sampler(X_train, y_train)

In [9]:
from sklearn.model_selection import KFold as KF, GridSearchCV
cv_outer = KF(n_splits = 10, shuffle=True, random_state=42)
cv_inner = KF(n_splits = 5, shuffle=True, random_state=42)


In [10]:


def balanced_log_loss(y_true, y_pred):
    # y_true: correct labels 0, 1
    # y_pred: predicted probabilities of class=1
    # calculate the number of observations for each class
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    # calculate the weights for each class to balance classes
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    # calculate the predicted probabilities for each class
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    # calculate the summed log loss for each class
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    # calculate the weighted summed logarithmic loss
    # (factgor of 2 included to give same result as LL with balanced input)
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    # return the average log loss
    return balanced_log_loss/(N_0+N_1)



In [11]:

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from imblearn.over_sampling import RandomOverSampler
import xgboost
from tabpfn import TabPFNClassifier
from tqdm.notebook import tqdm

class Ensemble():
    def __init__(self):
        self.imputer = SimpleImputer(missing_values=np.nan, strategy='median')
        self.classifiers =[xgboost.XGBClassifier(), TabPFNClassifier(model_path=".pretrained/tabpfn-v2.5-classifier-v2.5_default.ckpt")]
    
    def fit(self,X,y):
        y = y.values
        unique_classes, y = np.unique(y, return_inverse=True)
        self.classes_ = unique_classes
        X = self.imputer.fit_transform(X)
#         X = normalize(X,axis=0)
        for classifier in self.classifiers:
            if classifier==self.classifiers[1]:
                classifier.fit(X,y)
            else :
                classifier.fit(X, y)
     
    def predict_proba(self, x):
        x = self.imputer.transform(x)
#         x = normalize(x,axis=0)
        probabilities = np.stack([classifier.predict_proba(x) for classifier in self.classifiers])
        averaged_probabilities = np.mean(probabilities, axis=0)
        class_0_est_instances = averaged_probabilities[:, 0].sum()
        others_est_instances = averaged_probabilities[:, 1:].sum()
        # Weighted probabilities based on class imbalance
        new_probabilities = averaged_probabilities * np.array([[1/(class_0_est_instances if i==0 else others_est_instances) for i in range(averaged_probabilities.shape[1])]])
        return new_probabilities / np.sum(new_probabilities, axis=1, keepdims=1) 


In [22]:
from tqdm.notebook import tqdm

from tqdm.notebook import tqdm
import numpy as np

def training(model, x, y):
    """
    Cross-validation training with balanced log loss.
    x : pandas DataFrame after resampling
    y : pandas Series/array after resampling
    """
    outer_results = []
    best_loss = np.inf
    best_model = None
    splits = 5

    for split, (train_idx, val_idx) in enumerate(tqdm(cv_inner.split(x), total=splits), 1):
        x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(x_train, y_train)
        y_pred = model.predict_proba(x_val)

        # Convert multi-class probs to binary-like for log loss
        probabilities = np.concatenate((y_pred[:, :1], np.sum(y_pred[:, 1:], axis=1, keepdims=True)), axis=1)
        p0 = probabilities[:, :1]
        p0[p0 > 0.86] = 1
        p0[p0 < 0.14] = 0

        y_p = np.where(p0 >= 0.5, 0, 1).flatten()
        loss = balanced_log_loss(y_val, y_p)

        if loss < best_loss:
            best_model = model
            best_loss = loss
            print(f'best_model_saved at split {split}')

        outer_results.append(loss)
        print(f'> val_loss={loss:.5f}, split={split}')

    print(f'Mean LOSS: {np.mean(outer_results):.5f}')
    return best_model

In [13]:
from datetime import datetime
greeks = pd.read_csv('../greeks.csv')
times = greeks.Epsilon.copy()
times[greeks.Epsilon != 'Unknown'] = greeks.Epsilon[greeks.Epsilon != 'Unknown'].map(lambda x: datetime.strptime(x,'%m/%d/%Y').toordinal())
times[greeks.Epsilon == 'Unknown'] = np.nan

In [14]:
train_pred_and_time = pd.concat((X_train, times), axis=1)
test_predictors = X_test
test_pred_and_time = np.concatenate((test_predictors, np.zeros((len(test_predictors), 1)) + train_pred_and_time.Epsilon.max() + 1), axis=1)


In [ ]:
ros = RandomOverSampler(random_state=42)

train_ros, y_ros = ros.fit_resample(train_pred_and_time, greeks.Alpha)
print('Original dataset shape')
print(greeks.Alpha.value_counts())
print('Resample dataset shape')
print( y_ros.value_counts())





Original dataset shape
Alpha
A    509
B     61
G     29
D     18
Name: count, dtype: int64
Resample dataset shape
Alpha
B    509
A    509
D    509
G    509
Name: count, dtype: int64


In [19]:


x_ros = train_ros
y_ = y_train.Class



In [20]:
yt = Ensemble()

In [24]:
m = training(yt,x_ros,y_ros)

  0%|          | 0/5 [00:00<?, ?it/s]

TypeError: unsupported operand type(s) for -: 'int' and 'str'

In [7]:
# -------------------------------
# Full Pipeline: Load, Balance, Train Ensemble
# -------------------------------

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from imblearn.over_sampling import RandomOverSampler
import xgboost
from tabpfn import TabPFNClassifier
from tqdm.notebook import tqdm

# -------------------------------
# Load Data
# -------------------------------
X_train = pd.read_csv('../data/X_train_encoded.csv')
X_test  = pd.read_csv('../data/X_test_encoded.csv')
y_train = pd.read_csv('../data/y_train.csv')

# Flatten target
target_col = config.Target  # change to your target column name
y_train_numeric = y_train[target_col].values.flatten()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of classes:", len(np.unique(y_train_numeric)))

# -------------------------------
# Handle categorical columns (binary example)
# -------------------------------
# Convert 'EJ' to 0/1 if exists
if 'EJ' in X_train.columns:
    first_category = X_train.EJ.unique()[0]
    X_train.EJ = X_train.EJ.eq(first_category).astype(int)
    X_test.EJ  = X_test.EJ.eq(first_category).astype(int)

# -------------------------------
# Balance Training Data using RandomOverSampler
# -------------------------------
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train_numeric)

print("Original class distribution:")
print(pd.Series(y_train_numeric).value_counts())
print("Balanced class distribution:")
print(pd.Series(y_train_bal).value_counts())

# -------------------------------
# Ensemble Classifier: XGBoost + TabPFN
# -------------------------------
class Ensemble:
    def __init__(self):
        self.imputer = SimpleImputer(strategy='median')
        self.classifiers = [
            xgboost.XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
            TabPFNClassifier(model_path=".pretrained/tabpfn-v2.5-classifier-v2.5_default.ckpt")
        ]

    def fit(self, X, y):
        y = np.array(y).flatten()
        X = self.imputer.fit_transform(X)
        for clf in self.classifiers:
            if isinstance(clf, TabPFNClassifier):
                clf.fit(X, y)
            else:
                clf.fit(X, y)

    def predict_proba(self, X):
        X = self.imputer.transform(X)
        probs = np.stack([clf.predict_proba(X) for clf in self.classifiers])
        avg_probs = np.mean(probs, axis=0)
        # Normalize to sum 1
        return avg_probs / avg_probs.sum(axis=1, keepdims=True)

# -------------------------------
# Cross-Validation Training
# -------------------------------
cv_inner = KFold(n_splits=5, shuffle=True, random_state=42)

def training(model, X, y):
    best_loss = np.inf
    best_model = None
    losses = []

    X = pd.DataFrame(X)  # ensure DataFrame for iloc
    y = pd.Series(y)     # ensure Series for iloc

    for train_idx, val_idx in tqdm(cv_inner.split(X), total=cv_inner.get_n_splits()):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict_proba(X_val_fold)[:, 1]  # probability of positive class

        loss = log_loss(y_val_fold, y_pred)
        losses.append(loss)

        if loss < best_loss:
            best_loss = loss
            best_model = model

    print(f"Mean CV loss: {np.mean(losses):.5f}")
    return best_model

# -------------------------------
# Train Ensemble with CV
# -------------------------------
ensemble_model = Ensemble()
best_model = training(ensemble_model, X_train_bal, y_train_bal)

# -------------------------------
# Predict on Test Set
# -------------------------------
y_test_pred = best_model.predict_proba(X_test)[:, 1]
print("Test predictions shape:", y_test_pred.shape)

X_train shape: (617, 57)
X_test shape: (5, 58)
Number of classes: 2
Original class distribution:
0    509
1    108
Name: count, dtype: int64
Balanced class distribution:
1    509
0    509
Name: count, dtype: int64


  0%|          | 0/5 [00:00<?, ?it/s]

tabpfn-v2.5-classifier-v2.5_default.ckpt:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

Mean CV loss: 0.03337


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Id


In [31]:


x_ros = train_ros
y_ = y_train.Class



In [32]:
yt = Ensemble()

TypeError: TabPFNClassifier.__init__() got an unexpected keyword argument 'N_ensemble_configurations'